# At times, code will be reaped from the project

### Below we have the manual implementation of the Gravity theory of trade

Used in 02_EDA in section 7.

In [ ]:
# In an attempt to synthesise the missing data in the dataframe, we can use the Gravity formula of trade.
# For more notes, see notes_for_work.md (03/04/2026)
#
# Question: Using pure raw gdp_o and gdp_d leads to very unbalanced synthetic data. Might be worth trying gdpcap_ppp_o


df = ecowas_df_full.copy()

# So to train our model, we must remove all NaNs. Don't want it to learn from what's missing, obviously.
train_df = df.dropna(subset=["tradeflow_baci", "gdpcap_ppp_o", "gdpcap_ppp_d", "dist", "pop_o", "pop_d"])

# Logs for variables included in PPML
train_df["ln_gdp_o"] = np.log(train_df["gdpcap_ppp_o"])
train_df["ln_gdp_d"] = np.log(train_df["gdpcap_ppp_d"])
train_df["ln_dist"]  = np.log(train_df["dist"])
train_df["ln_pop_o"] = np.log(train_df["pop_o"])
train_df["ln_pop_d"] = np.log(train_df["pop_d"])


# We want to include fixed effects. We use the pyfixest library:
# Since we currently do not have fixed effects, this is just a standard Pooled Poisson Pseudo-Maximum Likelihood regression (PPML)
res = fepois(
    fml="tradeflow_baci ~ ln_gdp_o + ln_gdp_d + ln_dist + sibling + ln_pop_o + ln_pop_d",        # Here we can add Fixed Effects by adding a | after the coefficients
    data=train_df,
    )

print(res.summary())



#### BELOW WE HAVE MANUAL IMPLEMENTATION OF GRAVITY TRADE MODEL
# Not necessary, since we have an automatic predict

# Here we can extract the coefficients. 
coefs = res.coef()

beta1 = coefs["ln_gdp_o"]
beta2 = coefs["ln_gdp_d"]
beta3 = -coefs["ln_dist"]  # negative because distance reduces trade

# Below we save the effects of the fixed effects in a dict
#fixef = res.fixef()

# We also need to figure out which fixed effects to include as part of this synthesising of data. 
# These fixed effects should make the synthetic data more precise, by adding other variables to map some of the complexity of the involved countries 
# 
numerator   = train_df["tradeflow_baci"].mean()

denominator = ((train_df["gdp_o"]**beta1) *
               (train_df["gdp_d"]**beta2) /
               (train_df["dist"]**beta3)).mean()

G = numerator / denominator
print(f"Our scaling constant G is: {G}")

df["ln_gdp_o"] = np.log(df["gdp_o"])
df["ln_gdp_d"] = np.log(df["gdp_d"])
df["ln_dist"]  = np.log(df["dist"])

df["synthetic_trade"] = (
    G *
    (df["gdp_o"] ** beta1) *
    (df["gdp_d"] ** beta2) /
    (df["dist"]  ** beta3)
)

# Optional: add noise (gamma noise works well for multiplicative models)
noise = np.random.gamma(shape=1.0, scale=0.15, size=len(df))
df["synthetic_trade_noisy"] = df["synthetic_trade"] * (1 + noise)




### Below we have an attempt to use multiple variables to synthesise the baci trade data (so all tradeflows are included)

Used in 03_synthetic_data in section 1.

The model we would ideally use for synthetic data (In this case tradeflow_baci):

E[BACI | X] = exp(
    β₀ + β₁ ln GDP_o + β₂ ln GDP_d + β₃ ln dist + β₄ contig + γ₁ ln Comtrade_o + γ₂ ln Comtrade_d + γ₃ ln IMF_o + γ₄ ln IMF_d
)

We would only keep the y_variables where they are present


In [ ]:
def safe_poisson_draw(lam, threshold=5e5):
    """
    Draw from Poisson when lambda is small,
    use Normal approximation when lambda is large.

    Added due to errors with values that are too high in the augmented values

    """
    lam = np.asarray(lam, dtype=float)
    out = np.empty_like(lam)

    small = lam <= threshold
    large = lam > threshold

    # Exact Poisson where safe
    out[small] = np.random.poisson(lam[small])

    # Normal approximation where Poisson breaks
    out[large] = np.maximum(
        0,
        np.random.normal(
            loc=lam[large],
            scale=np.sqrt(lam[large])
        )
    )

    return out


# We take it step by step. First we want to create a baseline model (that is, using just the β's from above)
# We want it to work for ALL countries
df_work = df.copy()

# We ensure positive values for the gravity variables
gravity_mask = (
    (df_work["gdp_o"] > 0) &
    (df_work["gdp_d"] > 0) &
    (df_work["distw_arithmetic"] > 0)
)

df_work = df_work.loc[gravity_mask].copy()

df_work["ln_gdp_o"] = np.log(df_work["gdp_o"])
df_work["ln_gdp_d"] = np.log(df_work["gdp_d"])
df_work["ln_distw_arithmetic"] = np.log(df_work["distw_arithmetic"])

mask_observations = df_work["tradeflow_baci"].notna()

X_base = sm.add_constant(df_work.loc[mask_observations, ["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"]])
y_base = df_work.loc[mask_observations, "tradeflow_baci"]

ppml_base = sm.GLM(
    y_base,
    X_base,
    family=sm.families.Poisson()
).fit(cov_type="HC1")

# Then we can predict for all base rows that we have
X_all_base = sm.add_constant(df_work[["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"]])

mu_base_hat = ppml_base.predict(X_all_base)
df_work["baci_mu_base"] = mu_base_hat



# Now we can add the auxiliary variables
aux_vars = [
    "tradeflow_comtrade_o",
    "tradeflow_comtrade_d",
    "tradeflow_imf_o",
    "tradeflow_imf_d"
]

for v in aux_vars:
    df_work[f"ln_{v}"] = np.where(
        df_work[v] > 0,
        np.log(df_work[v]),
        np.nan
    )


# We have an error - if any of the auxiliary variables have a single positive line but the rest are missing, we will get a line that contains NaNs or infinite values that break the statsmodel
# -> exog can not contain any NaNs or inf, or it breaks!
# Let's first define our augmented variables - Then we update the mask below so that any missing values fails the entire row
aug_vars = ["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"] + [
    f"ln_{v}" for v in aux_vars
]

# All right! Time to look at the augmented model (where applicable)
mask_aug = (
    df_work["tradeflow_baci"].notna() &
    #df_work[[f"ln_{v}" for v in aux_vars]].notna().any(axis=1)       This is the problematic line. This adds rows, even if there are some unusable values
    np.isfinite(df_work[aug_vars]).all(axis=1)
)



'''
X_aug = sm.add_constant(
    df_work.loc[mask_aug,
                ["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"] + [f"ln_{v}" for v in aux_vars]
                ]
)
'''
X_aug = sm.add_constant(df_work.loc[mask_aug, aug_vars])

y_aug = df_work.loc[mask_aug, "tradeflow_baci"]

ppml_aug = sm.GLM(
    y_aug,
    X_aug,
    family=sm.families.Poisson()
).fit(cov_type="HC1")

# We predict an augmented mean 
X_all_aug = sm.add_constant(
    df_work[["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"] + [f"ln_{v}" for v in aux_vars]]
)

mu_aug_hat = ppml_aug.predict(X_all_aug)
# we can save it as a column to use:
df_work["baci_mu_aug"] = mu_aug_hat

# We need to predict which prediction to use (so basically, augmented when possible, otherwise use the baseline)
has_any_aux = df_work[[f"ln_{v}" for v in aux_vars]].notna().any(axis=1)
has_all_aux = np.isfinite(df_work[[f"ln_{v}" for v in aux_vars]]).all(axis=1)

df_work["baci_mu_aug_final"] = np.where(
    has_any_aux,
    df_work["baci_mu_aug"],
    df_work["baci_mu_base"]
)

# The values got too big for PPML (but only slightly), so we need to implement a cap
# To inspect the values, you can see here: df_work["baci_mu_aug_final"].describe(percentiles=[0.9, 0.95, 0.99, 0.999])

cap = df_work["baci_mu_aug_final"].quantile(0.99)
df_work["baci_mu_aug_capped"] = np.clip(
    df_work["baci_mu_aug_final"],
    a_min = 0,
    a_max = cap
)

# Now we can synthesise it once and for all!
missing_baci = df_work["tradeflow_baci"].isna()

df_work.loc[missing_baci, "tradeflow_baci_synth"] = safe_poisson_draw(
    df_work.loc[missing_baci, "baci_mu_aug_capped"].values
)

df_work.loc[~missing_baci, "tradeflow_baci_synth"] = (
    df_work.loc[~missing_baci, "tradeflow_baci"]
)

df_work["baci_source"] = "observed"

df_work.loc[missing_baci & has_any_aux, "baci_source"] = "imputed_gravity+trade"
df_work.loc[missing_baci & ~has_any_aux, "baci_source"] = "imputed_gravity_only"
